# Spendly independent generalization test

This notebook answers the next question: **do the frozen model choices work on new users,
a longer time period, changed financial behaviour, and anomaly patterns not present in the
original generator?**

It does not tune against the new stress-test labels and does not modify the application.
The original dataset trains the frozen models; a newly generated 12-month dataset is used
only as an external evaluation set.

## How to run

1. Open this notebook in Google Colab and run all cells.
2. Upload the original `combined_finance_raw.csv` when prompted.
3. The default creates 400 completely new users over 365 days. A CPU runtime is sufficient.
4. Download `spendly_generalization_output.zip` at the end.
5. Paste the block between `SPENDLY_GENERALIZATION_RESULT_START` and
   `SPENDLY_GENERALIZATION_RESULT_END` back into the task.

Expected runtime is several minutes. The acceptance thresholds are project targets rather
than universal financial-industry standards.

In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn",
    "matplotlib": "matplotlib", "seaborn": "seaborn", "joblib": "joblib",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependency check: PASS")

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import time
import warnings
import zipfile
from collections import defaultdict, deque
from datetime import datetime, time as clock_time, timedelta
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)
from sklearn.base import clone
from sklearn.ensemble import (
    ExtraTreesRegressor, GradientBoostingRegressor,
    HistGradientBoostingRegressor, IsolationForest,
)
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import (
    confusion_matrix, mean_squared_error, precision_recall_curve,
    precision_recall_fscore_support, r2_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")

SEED = 42
LOOKBACK = 8
STRESS_SEED = int(os.environ.get("SPENDLY_STRESS_SEED", "20260825"))
STRESS_USER_COUNT = int(os.environ.get("SPENDLY_STRESS_USERS", "400"))
STRESS_START = pd.Timestamp("2025-01-01")
STRESS_DAYS = 365
OUTPUT_DIR = Path("spendly_generalization_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

ACCEPTANCE = {
    "forecast_wape_max": 30.0,
    "forecast_r2_min": 0.50,
    "anomaly_precision_min": 0.60,
    "anomaly_recall_min": 0.75,
}
print({
    "stress_seed": STRESS_SEED, "stress_users": STRESS_USER_COUNT,
    "stress_days": STRESS_DAYS, "acceptance_targets": ACCEPTANCE,
})

In [ ]:
requested_path = os.environ.get("SPENDLY_DATA_PATH", "combined_finance_raw.csv")
ORIGINAL_DATA_PATH = Path(requested_path)
if not ORIGINAL_DATA_PATH.exists():
    matches = list(Path.cwd().rglob("combined_finance_raw.csv"))
    if matches:
        ORIGINAL_DATA_PATH = matches[0]
    else:
        try:
            from google.colab import files
            uploaded = files.upload()
            ORIGINAL_DATA_PATH = Path(next(iter(uploaded)))
        except Exception as exc:
            raise FileNotFoundError(
                "Upload the original combined_finance_raw.csv or set SPENDLY_DATA_PATH."
            ) from exc

ORIGINAL_SHA256 = hashlib.sha256(ORIGINAL_DATA_PATH.read_bytes()).hexdigest()
EXPECTED_SHA256 = "b39cb5e5725a504c8880358edd160945323e208625af441a8680decd0f34b834"
assert ORIGINAL_SHA256 == EXPECTED_SHA256, (
    "This is not the reviewed training dataset. Expected SHA256 " + EXPECTED_SHA256
)
raw = pd.read_csv(ORIGINAL_DATA_PATH, low_memory=False)
raw["transaction_timestamp"] = pd.to_datetime(raw["transaction_timestamp"], errors="raise")
raw["amount"] = pd.to_numeric(raw["amount"], errors="raise")
assert raw["transaction_id"].is_unique
print({
    "original_sha256": ORIGINAL_SHA256, "rows": len(raw),
    "users": raw["user_id"].nunique(),
    "start": str(raw["transaction_timestamp"].min()),
    "end": str(raw["transaction_timestamp"].max()),
})
print("Original training-data identity check: PASS")

## 1. Generate a structurally different stress dataset

This generator deliberately changes the data-generating process. It includes new users,
12 months instead of six, randomized recurring-payment days, missed payments, income shifts,
trend and seasonal changes, and both familiar and unseen anomaly scenarios.

A larger dataset made from the identical original rules would not be an independent test.

In [ ]:
STRESS_CATEGORIES = [
    "Airtime and Data", "Education", "Entertainment", "Food", "Healthcare",
    "Irregular Expenses", "Rent", "Savings", "Subscriptions", "Transport", "Utilities",
]
DISCRETIONARY_CATEGORIES = [
    "Airtime and Data", "Education", "Entertainment", "Food", "Healthcare",
    "Irregular Expenses", "Transport",
]
MERCHANTS = {
    "Airtime and Data": ["Safaricom", "Airtel", "Telkom"],
    "Education": ["Bookshop", "Course provider", "Campus store"],
    "Entertainment": ["Cinema", "Streaming", "Social outing"],
    "Food": ["Cafe", "Restaurant", "Supermarket", "Kibanda"],
    "Healthcare": ["Clinic", "Pharmacy", "Hospital"],
    "Irregular Expenses": ["Repair shop", "ATM", "Electronics store"],
    "Rent": ["Landlord transfer"], "Savings": ["Savings transfer"],
    "Subscriptions": ["Streaming", "Cloud service", "Gym"],
    "Transport": ["Matatu", "Fuel station", "Ride hailing"],
    "Utilities": ["KPLC", "Water company", "Gas supplier"],
}
SEEN_ANOMALIES = {
    "random_financial_shock", "unusually_large_category_expenditure",
    "sudden_transaction_frequency_increase", "duplicate_like_spending",
    "unexpected_recurring_expense_increase", "unusually_high_weekend_spending",
    "sudden_category_change",
}
UNSEEN_ANOMALIES = {
    "nighttime_cash_out", "merchant_hopping_burst", "split_transaction_evasion",
}

def income_band(income):
    if income < 30000:
        return "low"
    if income < 70000:
        return "middle"
    return "high"

def stress_base_amount(category, income, rng):
    scales = {
        "Airtime and Data": (0.002, 0.006), "Education": (0.004, 0.025),
        "Entertainment": (0.003, 0.016), "Food": (0.002, 0.012),
        "Healthcare": (0.003, 0.025), "Irregular Expenses": (0.010, 0.060),
        "Transport": (0.002, 0.010),
    }
    low, high = scales[category]
    return float(income * rng.lognormal(np.log((low + high) / 2), 0.45))

def generate_stress_dataset(user_count=400, start=STRESS_START, days=365, seed=STRESS_SEED):
    rng = np.random.default_rng(seed)
    records = []
    transaction_counter = 0

    def add_transaction(
        profile, timestamp, amount, category, transaction_type, merchant,
        payment_method, is_recurring=False, anomaly_type=None,
    ):
        nonlocal transaction_counter
        transaction_counter += 1
        records.append({
            "transaction_id": f"STRESS-T{transaction_counter:09d}",
            "user_id": profile["user_id"],
            "transaction_timestamp": pd.Timestamp(timestamp),
            "amount": round(max(1.0, float(amount)), 2),
            "currency": "KES", "category": category,
            "transaction_type": transaction_type, "merchant": merchant,
            "payment_method": payment_method, "is_recurring": bool(is_recurring),
            "source_dataset": "independent_stress_generator_v1", "is_synthetic": True,
            "user_age": profile["age"], "user_monthly_income": profile["base_income"],
            "user_employment_status": profile["status"],
            "user_income_day": profile["income_day"],
            "is_anomaly": anomaly_type is not None,
            "anomaly_type": anomaly_type,
            "label_source": "controlled_stress_label" if anomaly_type else None,
        })

    start_date = pd.Timestamp(start).date()
    end_date = start_date + timedelta(days=days - 1)
    seen = sorted(SEEN_ANOMALIES - {"random_financial_shock"})
    unseen = sorted(UNSEEN_ANOMALIES)

    for user_index in range(user_count):
        status = str(rng.choice(["student", "employed", "self_employed"], p=[0.36, 0.46, 0.18]))
        if status == "student":
            base_income = float(np.clip(rng.lognormal(np.log(26000), 0.38), 12000, 70000))
        elif status == "self_employed":
            base_income = float(np.clip(rng.lognormal(np.log(65000), 0.55), 18000, 220000))
        else:
            base_income = float(np.clip(rng.lognormal(np.log(52000), 0.40), 20000, 180000))
        profile = {
            "user_id": f"STRESS-U{user_index + 1:05d}", "age": int(rng.integers(18, 36)),
            "status": status, "base_income": base_income,
            "income_day": int(rng.choice([5, 15, 25, 28])),
            "rent_day": int(rng.integers(1, 6)), "utilities_day": int(rng.integers(8, 14)),
            "subscriptions_day": int(rng.integers(14, 19)), "savings_day": int(rng.integers(23, 29)),
        }
        rent = base_income * (0.16 if status == "student" else 0.23) * rng.uniform(0.80, 1.20)
        utilities = base_income * rng.uniform(0.03, 0.075)
        subscriptions = base_income * rng.uniform(0.006, 0.028)
        savings = base_income * rng.uniform(0.05, 0.20)
        base_daily_rate = 0.68 if status == "student" else 0.96
        annual_trend = float(rng.uniform(-0.04, 0.18))
        change_day = start_date + timedelta(days=int(rng.integers(120, 280)))
        income_change = float(rng.uniform(0.72, 1.30)) if rng.random() < 0.25 else 1.0
        schedule_shift = int(rng.choice([-2, -1, 1, 2])) if rng.random() < 0.20 else 0

        scenario_type = None
        scenario_day = None
        if rng.random() < 0.58:
            scenario_type = str(rng.choice(seen + unseen))
            scenario_day = start_date + timedelta(days=int(rng.integers(45, days - 10)))
            if scenario_type == "unusually_high_weekend_spending":
                while scenario_day.weekday() < 5:
                    scenario_day += timedelta(days=1)

        current = start_date
        while current <= end_date:
            progress = (current - start_date).days / max(1, days - 1)
            current_income = base_income * (income_change if current >= change_day else 1.0)
            inflation = 1.0 + annual_trend * progress
            weekend = current.weekday() >= 5
            shifted = schedule_shift if current >= change_day else 0

            if current.day == min(profile["income_day"], 28) and rng.random() > 0.06:
                add_transaction(
                    profile, datetime.combine(current, clock_time(8, 0)),
                    current_income * rng.normal(1.0, 0.08 if status != "self_employed" else 0.22),
                    "Income", "income", "Employer" if status != "self_employed" else "Client payment",
                    "Bank Transfer", True,
                )
            recurring_schedule = [
                (max(1, min(28, profile["rent_day"] + shifted)), rent, "Rent", "Landlord transfer", 0.97),
                (max(1, min(28, profile["utilities_day"] + shifted)), utilities * rng.normal(1, 0.12), "Utilities", str(rng.choice(MERCHANTS["Utilities"])), 0.93),
                (max(1, min(28, profile["subscriptions_day"] + shifted)), subscriptions, "Subscriptions", str(rng.choice(MERCHANTS["Subscriptions"])), 0.96),
                (max(1, min(28, profile["savings_day"] + shifted)), savings, "Savings", "Savings transfer", 0.82),
            ]
            for scheduled_day, recurring_amount, category, merchant, payment_probability in recurring_schedule:
                if current.day == scheduled_day and rng.random() < payment_probability:
                    amount = recurring_amount * inflation
                    anomaly = None
                    if scenario_type == "unexpected_recurring_expense_increase" and scenario_day:
                        if current.year == scenario_day.year and current.month == scenario_day.month and category == "Rent":
                            amount *= 1.65
                            anomaly = scenario_type
                    add_transaction(
                        profile, datetime.combine(current, clock_time(int(rng.integers(7, 20)), 0)),
                        amount, category, "expense", merchant,
                        "Bank Transfer" if category == "Savings" else "M-Pesa", True, anomaly,
                    )

            seasonal = 1.15 if current.month == 12 else 1.0
            daily_rate = base_daily_rate * (1.30 if weekend else 1.0)
            daily_count = int(rng.poisson(daily_rate))
            probabilities = np.array([0.08, 0.08 if status == "student" else 0.03, 0.12, 0.32, 0.07, 0.04, 0.29])
            probabilities = probabilities / probabilities.sum()
            for _ in range(daily_count):
                category = str(rng.choice(DISCRETIONARY_CATEGORIES, p=probabilities))
                amount = stress_base_amount(category, current_income, rng) * inflation * seasonal
                if weekend and category == "Entertainment":
                    amount *= 1.45
                if current.month in {1, 8} and category == "Education":
                    amount *= 1.65
                hour = int(rng.integers(6, 23))
                minute = int(rng.integers(0, 60))
                add_transaction(
                    profile, datetime.combine(current, clock_time(hour, minute)), amount,
                    category, "expense", str(rng.choice(MERCHANTS[category])),
                    str(rng.choice(["M-Pesa", "Debit Card", "Cash", "Bank Transfer"], p=[0.48, 0.24, 0.22, 0.06])),
                )

            if rng.random() < 1 / 260:
                add_transaction(
                    profile, datetime.combine(current, clock_time(16, int(rng.integers(0, 60)))),
                    current_income * rng.uniform(0.16, 0.48), "Irregular Expenses", "expense",
                    str(rng.choice(MERCHANTS["Irregular Expenses"])), "M-Pesa", False,
                    "random_financial_shock",
                )

            if scenario_day == current and scenario_type and scenario_type != "unexpected_recurring_expense_increase":
                if scenario_type == "unusually_large_category_expenditure":
                    category = str(rng.choice(["Food", "Entertainment", "Education"]))
                    add_transaction(profile, datetime.combine(current, clock_time(14, 20)), current_income * rng.uniform(0.42, 0.82), category, "expense", str(rng.choice(MERCHANTS[category])), "M-Pesa", False, scenario_type)
                elif scenario_type == "sudden_transaction_frequency_increase":
                    for event in range(7):
                        add_transaction(profile, datetime.combine(current, clock_time(12, event * 7)), rng.uniform(80, 650), "Food", "expense", "Cafe", "M-Pesa", False, scenario_type)
                elif scenario_type == "duplicate_like_spending":
                    duplicate_amount = round(rng.uniform(500, 2800), 2)
                    for minute_offset in [0, 2]:
                        add_transaction(profile, datetime.combine(current, clock_time(14, 10 + minute_offset)), duplicate_amount, "Food", "expense", "Restaurant", "M-Pesa", False, scenario_type)
                elif scenario_type == "unusually_high_weekend_spending":
                    for event in range(3):
                        add_transaction(profile, datetime.combine(current, clock_time(17 + event, 10)), current_income * rng.uniform(0.07, 0.15), "Entertainment", "expense", "Social outing", "M-Pesa", False, scenario_type)
                elif scenario_type == "sudden_category_change":
                    category = "Healthcare" if status == "student" else "Education"
                    add_transaction(profile, datetime.combine(current, clock_time(13, 15)), current_income * rng.uniform(0.16, 0.34), category, "expense", str(rng.choice(MERCHANTS[category])), "M-Pesa", False, scenario_type)
                elif scenario_type == "nighttime_cash_out":
                    for event in range(3):
                        add_transaction(profile, datetime.combine(current, clock_time(2, 5 + event * 8)), current_income * rng.uniform(0.06, 0.14), "Irregular Expenses", "expense", "ATM", "Cash", False, scenario_type)
                elif scenario_type == "merchant_hopping_burst":
                    merchants = ["Cafe", "Restaurant", "Supermarket", "Kibanda", "Food delivery"]
                    for event, merchant in enumerate(merchants):
                        add_transaction(profile, datetime.combine(current, clock_time(19, event * 9)), rng.uniform(200, 1200), "Food", "expense", merchant, "M-Pesa", False, scenario_type)
                elif scenario_type == "split_transaction_evasion":
                    for event in range(4):
                        add_transaction(profile, datetime.combine(current, clock_time(11, 5 + event * 11)), current_income * rng.uniform(0.055, 0.085), "Irregular Expenses", "expense", "Electronics store", "Debit Card", False, scenario_type)
            current += timedelta(days=1)

    stress = pd.DataFrame(records).sort_values(
        ["user_id", "transaction_timestamp", "transaction_id"]
    ).reset_index(drop=True)
    return stress

In [ ]:
started = time.perf_counter()
stress = generate_stress_dataset(
    user_count=STRESS_USER_COUNT, start=STRESS_START, days=STRESS_DAYS, seed=STRESS_SEED
)
STRESS_DATA_PATH = OUTPUT_DIR / "independent_stress_finance.csv"
stress.to_csv(STRESS_DATA_PATH, index=False)
stress_sha256 = hashlib.sha256(STRESS_DATA_PATH.read_bytes()).hexdigest()
stress["transaction_timestamp"] = pd.to_datetime(stress["transaction_timestamp"])
stress_summary = {
    "sha256": stress_sha256, "rows": int(len(stress)),
    "users": int(stress["user_id"].nunique()),
    "start": str(stress["transaction_timestamp"].min()),
    "end": str(stress["transaction_timestamp"].max()),
    "anomaly_rows": int(stress["is_anomaly"].sum()),
    "seen_anomaly_rows": int(stress["anomaly_type"].isin(SEEN_ANOMALIES).sum()),
    "unseen_anomaly_rows": int(stress["anomaly_type"].isin(UNSEEN_ANOMALIES).sum()),
    "generation_seconds": time.perf_counter() - started,
}
assert stress["transaction_id"].is_unique
assert stress["amount"].gt(0).all()
assert stress["user_id"].nunique() == STRESS_USER_COUNT
assert stress["transaction_timestamp"].max() - stress["transaction_timestamp"].min() >= pd.Timedelta(days=360)
assert stress_summary["seen_anomaly_rows"] > 0 and stress_summary["unseen_anomaly_rows"] > 0
display(pd.Series(stress_summary, name="value").to_frame())
display(stress.loc[stress["is_anomaly"], "anomaly_type"].value_counts().rename("rows").to_frame())
print("Independent stress-data quality checks: PASS")

monthly_audit = (
    stress.assign(month=stress["transaction_timestamp"].dt.to_period("M").astype(str))
    .groupby("month").agg(
        transactions=("transaction_id", "size"),
        expense_kes=("amount", lambda values: float(values.sum())),
        anomalies=("is_anomaly", "sum"),
    ).reset_index()
)
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
sns.lineplot(data=monthly_audit, x="month", y="transactions", marker="o", ax=axes[0], color="#2563eb")
axes[0].set_title("Stress dataset transaction coverage by month")
axes[0].tick_params(axis="x", rotation=45)
sns.barplot(data=monthly_audit, x="month", y="anomalies", ax=axes[1], color="#dc2626")
axes[1].set_title("Controlled anomaly coverage by month")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 2. Freeze and externally test the forecast

Hyperparameters and feature logic come from the optimized notebook. The model is fitted on
the original reviewed dataset and applied directly to the new users. No stress-set week is
used for model fitting, calibration, or selection.

In [ ]:
def bools(series: pd.Series) -> pd.Series:
    return series.fillna(False).astype(str).str.lower().isin({"true", "1", "yes", "y", "t"})


def load_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    data = pd.read_csv(CSV, low_memory=False)
    data["transaction_timestamp"] = pd.to_datetime(data["transaction_timestamp"])
    data["amount"] = pd.to_numeric(data["amount"])
    data["is_recurring"] = bools(data["is_recurring"])
    data["transaction_type"] = data["transaction_type"].astype(str).str.lower()
    data["category"] = data["category"].astype(str).str.strip()
    profiles = (
        data[["user_id", "user_age", "user_monthly_income", "user_employment_status", "user_income_day"]]
        .drop_duplicates("user_id")
        .set_index("user_id")
    )
    return data, profiles


def build_weekly(data: pd.DataFrame, profiles: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    working = data.copy()
    working["period_start"] = working["transaction_timestamp"].dt.to_period("W-SUN").dt.start_time
    expense = working[working["transaction_type"].eq("expense")].copy()
    income = working[working["transaction_type"].eq("income")].copy()
    expense["recurring_amount"] = np.where(expense["is_recurring"], expense["amount"], 0.0)
    base = expense.groupby(["user_id", "period_start"]).agg(
        total_spending=("amount", "sum"),
        transaction_count=("transaction_id", "size"),
        average_transaction_amount=("amount", "mean"),
        recurring_expenses=("recurring_amount", "sum"),
    ).reset_index()
    category = expense.pivot_table(
        index=["user_id", "period_start"], columns="category", values="amount", aggfunc="sum", fill_value=0.0
    ).rename(columns=lambda value: "category_" + str(value).lower().replace(" ", "_")).reset_index()
    category_columns = sorted(c for c in category if c.startswith("category_"))
    income_agg = income.groupby(["user_id", "period_start"])["amount"].sum().rename("income").reset_index()
    grid = pd.MultiIndex.from_product(
        [sorted(data["user_id"].unique()), pd.date_range(working["period_start"].min(), working["period_start"].max(), freq="W-MON")],
        names=["user_id", "period_start"],
    ).to_frame(index=False)
    weekly = grid.merge(base, how="left", on=["user_id", "period_start"]).merge(
        category, how="left", on=["user_id", "period_start"]
    ).merge(income_agg, how="left", on=["user_id", "period_start"])
    numeric = ["total_spending", "transaction_count", "average_transaction_amount", "recurring_expenses", "income", *category_columns]
    weekly[numeric] = weekly[numeric].fillna(0.0)
    weekly = weekly.join(profiles, on="user_id")
    weekly = weekly.sort_values(["user_id", "period_start"]).reset_index(drop=True)
    grouped = weekly.groupby("user_id", group_keys=False)
    weekly["previous_period_spending"] = grouped["total_spending"].shift(1).fillna(0.0)
    weekly["rolling_average_4"] = grouped["total_spending"].transform(
        lambda values: values.shift(1).rolling(4, min_periods=1).mean()
    ).fillna(0.0)
    previous = weekly["previous_period_spending"]
    weekly["spending_growth_rate"] = np.where(
        previous > 0, (weekly["total_spending"] - previous) / previous, 0.0
    )
    weekly["spending_growth_rate"] = weekly["spending_growth_rate"].replace([np.inf, -np.inf], 0.0)
    features = [
        "total_spending", *category_columns, "transaction_count", "average_transaction_amount", "income",
        "recurring_expenses", "previous_period_spending", "rolling_average_4", "spending_growth_rate",
    ]
    return weekly, features


def day_in_week(period_start: pd.Timestamp, day: int) -> float:
    dates = pd.date_range(period_start, period_start + pd.Timedelta(days=6), freq="D")
    return float(any(d.day == day for d in dates))


def days_before_payday_in_week(period_start: pd.Timestamp, payday: int) -> float:
    dates = pd.date_range(period_start, period_start + pd.Timedelta(days=6), freq="D")
    return float(sum(((payday - int(d.day)) % 30) <= 3 for d in dates))


def nonzero_median(values: np.ndarray) -> np.ndarray:
    output = np.zeros(values.shape[0], dtype=float)
    for i, row in enumerate(values):
        positive = row[row > 0]
        output[i] = float(np.median(positive)) if len(positive) else 0.0
    return output


def build_sequences(weekly: pd.DataFrame, features: list[str]) -> dict[str, np.ndarray]:
    rows = []
    for user_id, group in weekly.groupby("user_id", sort=True):
        group = group.sort_values("period_start").reset_index(drop=True)
        values = group[features].to_numpy(float)
        for end in range(LOOKBACK - 1, len(group) - 1):
            target_idx = end + 1
            rows.append((
                str(user_id),
                values[end - LOOKBACK + 1:end + 1],
                float(group.loc[target_idx, "total_spending"]),
                pd.Timestamp(group.loc[target_idx, "period_start"]),
                float(group.loc[target_idx, "user_monthly_income"]),
                float(group.loc[target_idx, "user_age"]),
                str(group.loc[target_idx, "user_employment_status"]),
                int(group.loc[target_idx, "user_income_day"]),
            ))
    user = np.array([r[0] for r in rows])
    history = np.stack([r[1] for r in rows])
    target = np.array([r[2] for r in rows])
    dates = pd.to_datetime([r[3] for r in rows]).to_numpy()
    income = np.array([r[4] for r in rows])
    age = np.array([r[5] for r in rows])
    status = np.array([r[6] for r in rows])
    payday = np.array([r[7] for r in rows])

    total_idx = features.index("total_spending")
    recurring_categories = {
        "rent": features.index("category_rent"),
        "utilities": features.index("category_utilities"),
        "subscriptions": features.index("category_subscriptions"),
        "savings": features.index("category_savings"),
    }
    totals = history[:, :, total_idx]
    recurring_history = sum(history[:, :, idx] for idx in recurring_categories.values())
    discretionary = np.maximum(0.0, totals - recurring_history)
    target_context = []
    calendar_baseline = []
    for i, target_date in enumerate(pd.to_datetime(dates)):
        flags = {
            "rent": day_in_week(target_date, 2),
            "utilities": day_in_week(target_date, 10),
            "subscriptions": day_in_week(target_date, 15),
            "savings": day_in_week(target_date, 26),
        }
        recurring_estimate = 0.0
        for name, idx in recurring_categories.items():
            recurring_estimate += flags[name] * nonzero_median(history[i:i+1, :, idx])[0]
        discretionary_estimate = float(np.median(discretionary[i, -4:]))
        calendar_baseline.append(recurring_estimate + discretionary_estimate)
        month_angle = 2 * math.pi * (target_date.month - 1) / 12
        week_angle = 2 * math.pi * (target_date.isocalendar().week - 1) / 52
        target_context.append([
            income[i], age[i], payday[i],
            float(status[i] == "student"), float(status[i] == "employed"), float(status[i] == "self_employed"),
            flags["rent"], flags["utilities"], flags["subscriptions"], flags["savings"],
            days_before_payday_in_week(target_date, payday[i]),
            math.sin(month_angle), math.cos(month_angle), math.sin(week_angle), math.cos(week_angle),
            float(target_date.month), float(target_date.day),
            float(np.mean(totals[i, -4:])), float(np.median(totals[i, -4:])), float(np.std(totals[i, -4:])),
            float(np.mean(discretionary[i, -4:])), float(np.median(discretionary[i, -4:])),
            float(totals[i, -1] - totals[i, -4]),
            float(calendar_baseline[-1]),
        ])
    target_context = np.asarray(target_context, dtype=float)
    X = np.concatenate([history.reshape(len(history), -1), target_context], axis=1)
    return {
        "X": X, "history": history, "context": target_context,
        "y": target, "date": dates, "user": user, "income": income,
        "calendar_baseline": np.asarray(calendar_baseline),
    }


def forecast_metric_record(actual: np.ndarray, pred: np.ndarray) -> dict[str, float]:
    pred = np.maximum(0.0, np.asarray(pred))
    ae = np.abs(actual - pred)
    return {
        "mae": float(ae.mean()),
        "rmse": float(mean_squared_error(actual, pred) ** 0.5),
        "wape": float(100 * ae.sum() / np.abs(actual).sum()),
        "r2": float(r2_score(actual, pred)),
        "bias": float(np.mean(pred - actual)),
    }


def candidate_specs() -> dict[str, tuple[object, str]]:
    return {
        "linear_context": (LinearRegression(n_jobs=-1), "raw"),
        "ridge_10_raw": (make_pipeline(StandardScaler(), Ridge(alpha=10.0)), "raw"),
        "ridge_100_raw": (make_pipeline(StandardScaler(), Ridge(alpha=100.0)), "raw"),
        "ridge_10_log": (make_pipeline(StandardScaler(), Ridge(alpha=10.0)), "log"),
        "ridge_100_log": (make_pipeline(StandardScaler(), Ridge(alpha=100.0)), "log"),
        "ridge_10_residual": (make_pipeline(StandardScaler(), Ridge(alpha=10.0)), "residual"),
        "ridge_100_residual": (make_pipeline(StandardScaler(), Ridge(alpha=100.0)), "residual"),
        "ridge_10_income_ratio": (make_pipeline(StandardScaler(), Ridge(alpha=10.0)), "ratio"),
        "hist_poisson": (HistGradientBoostingRegressor(loss="poisson", learning_rate=0.05, max_iter=250, max_leaf_nodes=15, min_samples_leaf=30, l2_regularization=10.0, random_state=SEED), "raw"),
        "hist_squared": (HistGradientBoostingRegressor(loss="squared_error", learning_rate=0.05, max_iter=250, max_leaf_nodes=15, min_samples_leaf=30, l2_regularization=10.0, random_state=SEED), "raw"),
        "gbr_huber": (GradientBoostingRegressor(loss="huber", learning_rate=0.03, n_estimators=250, max_depth=2, min_samples_leaf=20, random_state=SEED), "raw"),
        "extra_trees": (ExtraTreesRegressor(n_estimators=300, min_samples_leaf=5, max_features=0.7, n_jobs=-1, random_state=SEED), "raw"),
    }


def fit_predict(spec: tuple[object, str], Xtr: np.ndarray, ytr: np.ndarray, btr: np.ndarray, itr: np.ndarray, Xva: np.ndarray, bva: np.ndarray, iva: np.ndarray) -> np.ndarray:
    estimator, transform = spec
    estimator = clone(estimator)
    if transform == "log":
        estimator.fit(Xtr, np.log1p(ytr))
        return np.expm1(estimator.predict(Xva))
    if transform == "residual":
        estimator.fit(Xtr, ytr - btr)
        return bva + estimator.predict(Xva)
    if transform == "ratio":
        estimator.fit(Xtr, ytr / np.maximum(itr, 1.0))
        return estimator.predict(Xva) * iva
    estimator.fit(Xtr, ytr)
    return estimator.predict(Xva)

In [ ]:
CSV = ORIGINAL_DATA_PATH
original_data, original_profiles = load_data()
original_weekly, forecast_features = build_weekly(original_data, original_profiles)
original_sequences = build_sequences(original_weekly, forecast_features)

stress_model_data = stress.copy()
stress_model_data["transaction_timestamp"] = pd.to_datetime(stress_model_data["transaction_timestamp"])
stress_model_data["is_recurring"] = bools(stress_model_data["is_recurring"])
stress_profiles = (
    stress_model_data[[
        "user_id", "user_age", "user_monthly_income",
        "user_employment_status", "user_income_day",
    ]].drop_duplicates("user_id").set_index("user_id")
)
stress_weekly, stress_forecast_features = build_weekly(stress_model_data, stress_profiles)
assert forecast_features == stress_forecast_features, "Forecast feature schema changed"
stress_sequences = build_sequences(stress_weekly, stress_forecast_features)

frozen_gbr = clone(candidate_specs()["gbr_huber"][0])
frozen_gbr.fit(original_sequences["X"], original_sequences["y"])
gbr_prediction = np.maximum(0.0, frozen_gbr.predict(stress_sequences["X"]))
calendar_prediction = np.maximum(0.0, stress_sequences["calendar_baseline"])
forecast_external_metrics = pd.DataFrame([
    {"model": "gbr_huber_frozen", **forecast_metric_record(stress_sequences["y"], gbr_prediction)},
    {"model": "calendar_baseline_frozen", **forecast_metric_record(stress_sequences["y"], calendar_prediction)},
]).sort_values("mae").reset_index(drop=True)
forecast_external_predictions = pd.DataFrame({
    "user_id": stress_sequences["user"],
    "target_week": pd.to_datetime(stress_sequences["date"]),
    "actual": stress_sequences["y"],
    "gbr_huber_prediction": gbr_prediction,
    "calendar_baseline_prediction": calendar_prediction,
})
display(forecast_external_metrics.round(3))

forecast_monthly = []
for month, part in forecast_external_predictions.groupby(forecast_external_predictions["target_week"].dt.to_period("M")):
    record = forecast_metric_record(part["actual"].to_numpy(), part["gbr_huber_prediction"].to_numpy())
    forecast_monthly.append({"month": str(month), **record, "samples": len(part)})
forecast_monthly = pd.DataFrame(forecast_monthly)
display(forecast_monthly.round(3))

primary_forecast = forecast_external_metrics[
    forecast_external_metrics["model"].eq("gbr_huber_frozen")
].iloc[0]
forecast_checks = {
    "wape_at_or_below_target": bool(primary_forecast["wape"] <= ACCEPTANCE["forecast_wape_max"]),
    "r2_at_or_above_target": bool(primary_forecast["r2"] >= ACCEPTANCE["forecast_r2_min"]),
}
forecast_pass = all(forecast_checks.values())
print({"forecast_checks": forecast_checks, "forecast_pass": forecast_pass})

fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
sns.lineplot(data=forecast_monthly, x="month", y="wape", marker="o", ax=axes[0], color="#2563eb")
axes[0].axhline(ACCEPTANCE["forecast_wape_max"], linestyle="--", color="#dc2626", label="Target")
axes[0].set_title("Frozen GBR forecast WAPE by month")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend()
sample_points = forecast_external_predictions.sample(min(2500, len(forecast_external_predictions)), random_state=SEED)
axes[1].scatter(sample_points["actual"], sample_points["gbr_huber_prediction"], alpha=0.25, s=12)
limit = max(sample_points["actual"].max(), sample_points["gbr_huber_prediction"].max())
axes[1].plot([0, limit], [0, limit], "--", color="black")
axes[1].set(title="Frozen GBR: actual vs predicted", xlabel="Actual weekly spend (KES)", ylabel="Predicted weekly spend (KES)")
plt.tight_layout()
plt.show()

## 3. Freeze and externally test anomaly detection

The scaler, Isolation Forest, rule configuration, hybrid weight, and threshold are learned or
fixed using only the original dataset. Stress labels are opened only after predictions exist.
Results are shown separately for familiar and unseen anomaly families.

In [ ]:
def safe_ratio(numerator: float, denominator: float, neutral: float = 0.0) -> float:
    return float(numerator / denominator) if denominator and np.isfinite(denominator) else neutral


def build_past_only_features(raw: pd.DataFrame) -> pd.DataFrame:
    expense = raw[raw["transaction_type"].astype(str).str.lower().eq("expense")].copy()
    expense["transaction_timestamp"] = pd.to_datetime(expense["transaction_timestamp"])
    expense["amount"] = pd.to_numeric(expense["amount"], errors="coerce")
    expense["is_recurring"] = bools(expense["is_recurring"])
    expense["is_anomaly_target"] = bools(expense["is_anomaly"])
    expense = expense.sort_values(["user_id", "transaction_timestamp", "transaction_id"]).reset_index(drop=True)

    engineered_rows: list[dict] = []
    for user_id, group in expense.groupby("user_id", sort=False):
        q1h: deque[tuple[pd.Timestamp, float]] = deque()
        q24h: deque[tuple[pd.Timestamp, float]] = deque()
        q7d: deque[tuple[pd.Timestamp, float]] = deque()
        sum24h = 0.0
        sum7d = 0.0
        user_amounts: list[float] = []
        category_amounts: dict[str, list[float]] = defaultdict(list)
        recurring_amounts: dict[str, list[float]] = defaultdict(list)
        category_counts: dict[str, int] = defaultdict(int)
        last_exact: dict[tuple[str, str, float], pd.Timestamp] = {}
        last_merchant_category: dict[tuple[str, str], pd.Timestamp] = {}
        last_timestamp: pd.Timestamp | None = None
        day_key = None
        day_count = 0
        day_sum = 0.0

        for row in group.itertuples(index=False):
            timestamp = pd.Timestamp(row.transaction_timestamp)
            amount = float(row.amount)
            category = str(row.category)
            merchant = str(row.merchant)
            income = float(row.user_monthly_income)
            current_day = timestamp.date()
            if day_key != current_day:
                day_key = current_day
                day_count = 0
                day_sum = 0.0

            cut1h = timestamp - pd.Timedelta(hours=1)
            cut24h = timestamp - pd.Timedelta(days=1)
            cut7d = timestamp - pd.Timedelta(days=7)
            while q1h and q1h[0][0] < cut1h:
                q1h.popleft()
            while q24h and q24h[0][0] < cut24h:
                sum24h -= q24h.popleft()[1]
            while q7d and q7d[0][0] < cut7d:
                sum7d -= q7d.popleft()[1]

            user_mean = float(np.mean(user_amounts)) if user_amounts else np.nan
            user_median = float(np.median(user_amounts)) if user_amounts else np.nan
            cat_values = category_amounts[category]
            category_mean = float(np.mean(cat_values)) if cat_values else np.nan
            category_median = float(np.median(cat_values)) if cat_values else np.nan
            recurring_values = recurring_amounts[category]
            recurring_median = float(np.median(recurring_values)) if recurring_values else np.nan
            exact_key = (merchant, category, round(amount, 2))
            merchant_key = (merchant, category)
            exact_age = (
                (timestamp - last_exact[exact_key]).total_seconds() / 60.0
                if exact_key in last_exact else np.nan
            )
            merchant_age = (
                (timestamp - last_merchant_category[merchant_key]).total_seconds() / 60.0
                if merchant_key in last_merchant_category else np.nan
            )
            past_count = len(user_amounts)

            engineered_rows.append({
                "transaction_id": row.transaction_id,
                "log_amount": np.log1p(max(0.0, amount)),
                "amount_to_income": safe_ratio(amount, income),
                "amount_to_user_mean": safe_ratio(amount, user_mean, 1.0),
                "amount_to_user_median": safe_ratio(amount, user_median, 1.0),
                "amount_to_category_mean": safe_ratio(amount, category_mean, 1.0),
                "amount_to_category_median": safe_ratio(amount, category_median, 1.0),
                "recurring_amount_ratio": safe_ratio(amount, recurring_median, 1.0),
                "past_user_count": past_count,
                "category_proportion": safe_ratio(category_counts[category], past_count),
                "previous_1h_count": len(q1h),
                "previous_24h_count": len(q24h),
                "previous_7d_count": len(q7d),
                "previous_24h_sum_to_income": safe_ratio(sum24h, income),
                "previous_7d_sum_to_income": safe_ratio(sum7d, income),
                "same_day_count_before": day_count,
                "same_day_sum_to_income": safe_ratio(day_sum, income),
                "time_since_previous_hours": (
                    (timestamp - last_timestamp).total_seconds() / 3600.0 if last_timestamp is not None else np.nan
                ),
                "same_exact_age_minutes": exact_age,
                "same_merchant_category_age_minutes": merchant_age,
                "is_recurring_feature": int(bool(row.is_recurring)),
                "is_weekend": int(timestamp.dayofweek >= 5),
                "hour_sin": np.sin(2 * np.pi * timestamp.hour / 24),
                "hour_cos": np.cos(2 * np.pi * timestamp.hour / 24),
            })

            q1h.append((timestamp, amount))
            q24h.append((timestamp, amount))
            q7d.append((timestamp, amount))
            sum24h += amount
            sum7d += amount
            user_amounts.append(amount)
            category_amounts[category].append(amount)
            category_counts[category] += 1
            if bool(row.is_recurring):
                recurring_amounts[category].append(amount)
            last_exact[exact_key] = timestamp
            last_merchant_category[merchant_key] = timestamp
            last_timestamp = timestamp
            day_count += 1
            day_sum += amount

    engineered = pd.DataFrame(engineered_rows)
    return expense.merge(engineered, on="transaction_id", how="left", validate="one_to_one")


def split_chronologically(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    unique_times = np.sort(frame["transaction_timestamp"].unique())
    train_count = max(1, int(np.floor(len(unique_times) * 0.70)))
    train_end = unique_times[train_count - 1]
    train = frame[frame["transaction_timestamp"] <= train_end].copy()
    held = frame[frame["transaction_timestamp"] > train_end].copy()
    held_times = np.sort(held["transaction_timestamp"].unique())
    val_count = max(1, int(np.floor(len(held_times) * 0.20)))
    val_end = held_times[val_count - 1]
    validation = held[held["transaction_timestamp"] <= val_end].copy()
    evaluation = held[held["transaction_timestamp"] > val_end].copy()
    return train, validation, evaluation


def anomaly_metric_record(actual: np.ndarray, predicted: np.ndarray) -> dict:
    actual = np.asarray(actual, dtype=bool)
    predicted = np.asarray(predicted, dtype=bool)
    tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=[False, True]).ravel()
    precision, recall, f1, _ = precision_recall_fscore_support(
        actual, predicted, average="binary", zero_division=0
    )
    return {
        "precision": float(precision), "recall": float(recall), "f1": float(f1),
        "false_positive_rate": float(fp / (fp + tn)), "tp": int(tp), "fp": int(fp),
        "tn": int(tn), "fn": int(fn), "alerts": int(predicted.sum()),
    }


def best_threshold(labels: np.ndarray, scores: np.ndarray) -> tuple[float, dict]:
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    if not len(thresholds):
        threshold = float(np.max(scores) + 1)
    else:
        f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
        threshold = float(thresholds[int(np.nanargmax(f1))])
    return threshold, anomaly_metric_record(labels, scores >= threshold)


def rules(frame: pd.DataFrame, config: dict) -> tuple[np.ndarray, np.ndarray]:
    duplicate = frame["same_exact_age_minutes"].between(0, config["duplicate_minutes"], inclusive="both")
    burst = frame["previous_1h_count"].ge(config["burst_count"])
    recurring = (
        frame["is_recurring_feature"].eq(1)
        & frame["category"].eq("Rent")
        & frame["recurring_amount_ratio"].ge(config["recurring_ratio"])
        & frame["past_user_count"].ge(20)
    )
    weekend = (
        frame["is_weekend"].eq(1)
        & frame["category"].eq("Entertainment")
        & frame["amount_to_income"].ge(config["weekend_ratio"])
    )
    random_shock = (
        frame["category"].eq("Irregular Expenses")
        & frame["amount_to_income"].ge(config["shock_ratio"])
    )
    extreme = frame["amount_to_income"].ge(config["extreme_ratio"])
    rare = (
        frame["past_user_count"].ge(30)
        & frame["category_proportion"].le(config["rare_proportion"])
        & frame["amount_to_income"].ge(config["rare_amount_ratio"])
    )
    flags = np.column_stack([duplicate, burst, recurring, weekend, random_shock, extreme, rare]).astype(int)
    weights = np.array([3.0, 2.0, 3.0, 2.0, 3.0, 3.0, 2.0])
    return flags.dot(weights), flags


def percentile_against(reference: np.ndarray, values: np.ndarray) -> np.ndarray:
    ordered = np.sort(np.asarray(reference, dtype=float))
    return np.searchsorted(ordered, values, side="right") / max(1, len(ordered))

In [ ]:
original_anomaly_frame = build_past_only_features(raw)
original_anomaly_train, original_anomaly_validation, _ = split_chronologically(original_anomaly_frame)
anomaly_features = [
    "log_amount", "amount_to_income", "amount_to_user_mean", "amount_to_user_median",
    "amount_to_category_mean", "amount_to_category_median", "recurring_amount_ratio",
    "category_proportion", "previous_1h_count", "previous_24h_count", "previous_7d_count",
    "previous_24h_sum_to_income", "previous_7d_sum_to_income", "same_day_count_before",
    "same_day_sum_to_income", "time_since_previous_hours", "same_exact_age_minutes",
    "same_merchant_category_age_minutes", "is_recurring_feature", "is_weekend", "hour_sin", "hour_cos",
]
for part in [original_anomaly_train, original_anomaly_validation]:
    part[anomaly_features] = part[anomaly_features].replace([np.inf, -np.inf], np.nan)
anomaly_fill_values = original_anomaly_train[anomaly_features].median().fillna(0.0)
frozen_anomaly_scaler = RobustScaler(quantile_range=(10, 90))
original_anomaly_x_train = frozen_anomaly_scaler.fit_transform(
    original_anomaly_train[anomaly_features].fillna(anomaly_fill_values)
)
original_anomaly_x_validation = frozen_anomaly_scaler.transform(
    original_anomaly_validation[anomaly_features].fillna(anomaly_fill_values)
)
original_validation_labels = original_anomaly_validation["is_anomaly_target"].to_numpy(bool)
frozen_if = IsolationForest(
    n_estimators=400, max_samples=256, max_features=1.0, contamination="auto",
    random_state=SEED, n_jobs=-1,
).fit(original_anomaly_x_train)
original_train_if_scores = -frozen_if.score_samples(original_anomaly_x_train)
original_validation_if_scores = -frozen_if.score_samples(original_anomaly_x_validation)
frozen_if_threshold, _ = best_threshold(original_validation_labels, original_validation_if_scores)

frozen_rule_config = {
    "duplicate_minutes": 5, "burst_count": 2, "recurring_ratio": 1.35,
    "weekend_ratio": 0.06, "shock_ratio": 0.15, "extreme_ratio": 0.30,
    "rare_proportion": 0.025, "rare_amount_ratio": 0.15,
}
frozen_rule_threshold = 1.0
frozen_hybrid_weight = 0.5
original_rule_validation_score, _ = rules(original_anomaly_validation, frozen_rule_config)
original_validation_if_percentile = percentile_against(
    original_train_if_scores, original_validation_if_scores
)
original_hybrid_validation_score = (
    original_rule_validation_score + frozen_hybrid_weight * original_validation_if_percentile
)
frozen_hybrid_threshold, original_hybrid_validation_metrics = best_threshold(
    original_validation_labels, original_hybrid_validation_score
)
print({
    "frozen_if_threshold": frozen_if_threshold,
    "frozen_hybrid_threshold": frozen_hybrid_threshold,
    "original_hybrid_validation_metrics": original_hybrid_validation_metrics,
})

In [ ]:
stress_anomaly_frame = build_past_only_features(stress_model_data)
stress_anomaly_frame[anomaly_features] = stress_anomaly_frame[anomaly_features].replace(
    [np.inf, -np.inf], np.nan
)
stress_anomaly_x = frozen_anomaly_scaler.transform(
    stress_anomaly_frame[anomaly_features].fillna(anomaly_fill_values)
)
stress_labels = stress_anomaly_frame["is_anomaly_target"].to_numpy(bool)
stress_if_scores = -frozen_if.score_samples(stress_anomaly_x)
stress_if_prediction = stress_if_scores >= frozen_if_threshold
stress_rule_scores, stress_rule_flags = rules(stress_anomaly_frame, frozen_rule_config)
stress_if_percentile = percentile_against(original_train_if_scores, stress_if_scores)
stress_hybrid_scores = stress_rule_scores + frozen_hybrid_weight * stress_if_percentile
stress_hybrid_prediction = stress_hybrid_scores >= frozen_hybrid_threshold

anomaly_external_metrics = pd.DataFrame([
    {"model": "isolation_forest_frozen", **anomaly_metric_record(stress_labels, stress_if_prediction)},
    {"model": "rules_plus_isolation_forest_frozen", **anomaly_metric_record(stress_labels, stress_hybrid_prediction)},
]).sort_values("f1", ascending=False).reset_index(drop=True)
display(anomaly_external_metrics.round(4))

anomaly_external_predictions = stress_anomaly_frame[[
    "transaction_id", "user_id", "transaction_timestamp", "amount", "category",
    "merchant", "anomaly_type", "is_anomaly_target",
]].copy()
anomaly_external_predictions["if_score"] = stress_if_scores
anomaly_external_predictions["if_prediction"] = stress_if_prediction
anomaly_external_predictions["hybrid_score"] = stress_hybrid_scores
anomaly_external_predictions["hybrid_prediction"] = stress_hybrid_prediction
anomaly_external_predictions["scenario_family"] = np.where(
    anomaly_external_predictions["anomaly_type"].isin(UNSEEN_ANOMALIES), "unseen",
    np.where(anomaly_external_predictions["anomaly_type"].isin(SEEN_ANOMALIES), "seen", "normal"),
)

anomaly_scenario_metrics = (
    anomaly_external_predictions[anomaly_external_predictions["is_anomaly_target"]]
    .groupby(["scenario_family", "anomaly_type"])["hybrid_prediction"]
    .agg(["size", "sum"]).reset_index()
    .rename(columns={"size": "labelled", "sum": "detected"})
)
anomaly_scenario_metrics["recall"] = (
    anomaly_scenario_metrics["detected"] / anomaly_scenario_metrics["labelled"]
)
display(anomaly_scenario_metrics.round(4))

primary_anomaly = anomaly_external_metrics[
    anomaly_external_metrics["model"].eq("rules_plus_isolation_forest_frozen")
].iloc[0]
anomaly_checks = {
    "precision_at_or_above_target": bool(primary_anomaly["precision"] >= ACCEPTANCE["anomaly_precision_min"]),
    "recall_at_or_above_target": bool(primary_anomaly["recall"] >= ACCEPTANCE["anomaly_recall_min"]),
}
anomaly_pass = all(anomaly_checks.values())
print({"anomaly_checks": anomaly_checks, "anomaly_pass": anomaly_pass})

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
scenario_plot = anomaly_scenario_metrics.sort_values("recall")
sns.barplot(data=scenario_plot, x="recall", y="anomaly_type", hue="scenario_family", ax=axes[0])
axes[0].axvline(ACCEPTANCE["anomaly_recall_min"], linestyle="--", color="#dc2626")
axes[0].set_xlim(0, 1)
axes[0].set_title("Frozen hybrid recall by stress scenario")
matrix = confusion_matrix(stress_labels, stress_hybrid_prediction, labels=[False, True])
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[1],
            xticklabels=["Normal", "Anomaly"], yticklabels=["Normal", "Anomaly"])
axes[1].set(title="Frozen hybrid confusion matrix", xlabel="Predicted", ylabel="Controlled label")
plt.tight_layout()
plt.show()

## 4. Decision and export

A pass means both target conditions are satisfied. A failure is useful: it identifies the
next development area without contaminating the stress set through repeated tuning.

In [ ]:
if forecast_pass and anomaly_pass:
    recommendation = "ready_for_guarded_app_integration"
    next_action = "Package GBR plus calendar fallback and the review-only anomaly hybrid."
elif not forecast_pass and not anomaly_pass:
    recommendation = "develop_both_models_before_app_integration"
    next_action = "Add inferred recurring schedules and drift-aware forecasting; redesign anomaly scoring for unseen patterns."
elif not forecast_pass:
    recommendation = "develop_forecast_before_app_integration"
    next_action = "Infer user-specific recurring schedules and add drift/regime features, then validate on a new untouched seed."
else:
    recommendation = "develop_anomaly_model_before_app_integration"
    next_action = "Improve precision and unseen-scenario coverage, then validate on a new untouched seed."

checks_table = pd.DataFrame([
    {"area": "dataset", "check": "new users", "status": bool(set(stress["user_id"]).isdisjoint(set(raw["user_id"])))},
    {"area": "dataset", "check": "12-month coverage", "status": True},
    {"area": "forecast", "check": f"WAPE <= {ACCEPTANCE['forecast_wape_max']}", "status": forecast_checks["wape_at_or_below_target"]},
    {"area": "forecast", "check": f"R2 >= {ACCEPTANCE['forecast_r2_min']}", "status": forecast_checks["r2_at_or_above_target"]},
    {"area": "anomaly", "check": f"precision >= {ACCEPTANCE['anomaly_precision_min']}", "status": anomaly_checks["precision_at_or_above_target"]},
    {"area": "anomaly", "check": f"recall >= {ACCEPTANCE['anomaly_recall_min']}", "status": anomaly_checks["recall_at_or_above_target"]},
    {"area": "governance", "check": "stress labels unused for tuning", "status": True},
    {"area": "governance", "check": "app unchanged", "status": True},
])
display(checks_table)
print({"recommendation": recommendation, "next_action": next_action})

forecast_external_metrics.to_csv(OUTPUT_DIR / "forecast_external_metrics.csv", index=False)
forecast_monthly.to_csv(OUTPUT_DIR / "forecast_monthly_metrics.csv", index=False)
forecast_external_predictions.to_csv(OUTPUT_DIR / "forecast_external_predictions.csv", index=False)
anomaly_external_metrics.to_csv(OUTPUT_DIR / "anomaly_external_metrics.csv", index=False)
anomaly_scenario_metrics.to_csv(OUTPUT_DIR / "anomaly_scenario_metrics.csv", index=False)
anomaly_external_predictions.to_csv(OUTPUT_DIR / "anomaly_external_predictions.csv", index=False)
checks_table.to_csv(OUTPUT_DIR / "generalization_checks.csv", index=False)
joblib.dump(frozen_gbr, OUTPUT_DIR / "frozen_gbr_huber.joblib")
joblib.dump(frozen_if, OUTPUT_DIR / "frozen_isolation_forest.joblib")
joblib.dump(frozen_anomaly_scaler, OUTPUT_DIR / "frozen_anomaly_scaler.joblib")

result_payload = {
    "notebook": "Spending_Model_Generalization_Test",
    "original_dataset_sha256": ORIGINAL_SHA256,
    "stress_dataset": stress_summary,
    "acceptance_targets": ACCEPTANCE,
    "forecast_external_metrics": forecast_external_metrics.round(6).to_dict(orient="records"),
    "forecast_checks": forecast_checks,
    "forecast_pass": forecast_pass,
    "anomaly_external_metrics": anomaly_external_metrics.round(6).to_dict(orient="records"),
    "anomaly_checks": anomaly_checks,
    "anomaly_pass": anomaly_pass,
    "scenario_recall": anomaly_scenario_metrics.round(6).to_dict(orient="records"),
    "overall_recommendation": recommendation,
    "next_action": next_action,
    "checks": {
        "new_users_only": bool(set(stress["user_id"]).isdisjoint(set(raw["user_id"]))),
        "stress_labels_used_for_tuning": False,
        "frozen_hyperparameters": True,
        "app_changed": False,
    },
}
results_path = OUTPUT_DIR / "spendly_generalization_results.json"
results_path.write_text(json.dumps(result_payload, indent=2, default=str), encoding="utf-8")

zip_path = Path("spendly_generalization_output.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(OUTPUT_DIR.parent))
print("SPENDLY_GENERALIZATION_RESULT_START")
print(json.dumps(result_payload, indent=2, default=str))
print("SPENDLY_GENERALIZATION_RESULT_END")
print("Saved:", zip_path.resolve())
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass

## Interpretation rule

Do not tune repeatedly against this exact stress dataset. If a model fails, make the indicated
development change and validate it on a new seed or, preferably, anonymized real transactions.
This preserves the stress dataset's value as independent evidence.